# Lesson 8: reproducible synthetic lab
Original fixed-seed strata, nearest-neighbor matching, and independent structural simulations for confounding, collider adjustment and mediation. Each equation is explicit and no external input is used.
All observations are generated for teaching; they do not describe actual people, firms or markets. Run every cell from a fresh kernel. Dependencies: numpy, pandas, scipy, scikit-learn.


In [1]:
"""Lesson 8. Synthetic DAGs and conditional comparisons; seed 8108.
Independent exogenous Gaussian draws implement the displayed structural equations.
No parameter or observation is fitted to an external source.
"""
import json
import numpy as np
rng=np.random.default_rng(8108)
def ols(y,*xs):
    X=np.column_stack([np.ones(len(y))]+list(xs));b=np.linalg.lstsq(X,y,rcond=None)[0];res=y-X@b
    return {'coef':b.tolist(),'r2':float(1-res@res/((y-y.mean())@(y-y.mean())))}
n=900;s=rng.choice(3,n,p=[.5,.3,.2]);x=rng.normal(size=n);e=np.array([.2,.5,.8])[s];d=rng.binomial(1,e)
y0=10+5*s+2*x+rng.normal(0,2,n);tau=2+s;y=y0+d*tau
rows=[]
for k in range(3):
    a=(s==k)&(d==1);b=(s==k)&(d==0)
    rows.append({'stratum':k,'n':int((s==k).sum()),'treated':int(a.sum()),'control':int(b.sum()),'mean_t':float(y[a].mean()),'mean_c':float(y[b].mean()),'gap':float(y[a].mean()-y[b].mean()),'truth':float(2+k)})
weights={key:np.array([r[field] for r in rows],float) for key,field in [('ATE','n'),('ATT','treated'),('ATU','control')]}
standardized={key:float(w@np.array([r['gap'] for r in rows])/w.sum()) for key,w in weights.items()}
truth={'ATE':float(tau.mean()),'ATT':float(tau[d==1].mean()),'ATU':float(tau[d==0].mean())}
matched=[];distance=[];used=[]
for i in np.where(d==1)[0]:
    candidates=np.where((d==0)&(s==s[i]))[0];j=candidates[np.argmin(abs(x[candidates]-x[i]))]
    matched.append(y[i]-y[j]);distance.append(abs(x[i]-x[j]));used.append(int(j))
m=6000
X=rng.normal(size=m);D=.9*X+rng.normal(size=m);Y=-D+3*X+rng.normal(size=m)
conf={'naive':ols(Y,D),'adjusted':ols(Y,D,X),'population_naive':-1+3*.9/(.9**2+1),'truth':-1,'cov_ratio':float(np.cov(D,X,ddof=0)[0,1]/np.var(D))}
D=rng.normal(size=m);U=rng.normal(size=m);Y=.6*D+1.5*U+rng.normal(size=m);C=D+U+.3*rng.normal(size=m);R=C+.5*rng.normal(size=m)
coll={'naive':ols(Y,D),'adjusted':ols(Y,D,C),'descendant':ols(Y,D,R),'truth':.6,'population_adjusted':.6-1.5/(1+.3**2)}
D=rng.normal(size=m);M=.8*D+rng.normal(size=m);Y=.5*D+1.5*M+rng.normal(size=m)
med={'total':ols(Y,D),'direct':ols(Y,D,M),'truth_total':.5+.8*1.5,'truth_direct':.5}
print(json.dumps({'seed':8108,'n':n,'strata':rows,'standardized':standardized,'truth':truth,'raw_gap':float(y[d==1].mean()-y[d==0].mean()),'weights':{key:(w/w.sum()).tolist() for key,w in weights.items()},'matching':{'att':float(np.mean(matched)),'mean_distance':float(np.mean(distance)),'max_distance':float(np.max(distance)),'treated_n':len(matched),'unique_controls':len(set(used))},'confounding':conf,'collider':coll,'mediation':med},indent=2,allow_nan=False))


{
  "seed": 8108,
  "n": 900,
  "strata": [
    {
      "stratum": 0,
      "n": 443,
      "treated": 105,
      "control": 338,
      "mean_t": 12.211713042322634,
      "mean_c": 10.04530672247428,
      "gap": 2.1664063198483543,
      "truth": 2.0
    },
    {
      "stratum": 1,
      "n": 286,
      "treated": 147,
      "control": 139,
      "mean_t": 18.174801837176968,
      "mean_c": 15.283715227086118,
      "gap": 2.8910866100908503,
      "truth": 3.0
    },
    {
      "stratum": 2,
      "n": 171,
      "treated": 138,
      "control": 33,
      "mean_t": 24.010106826265325,
      "mean_c": 20.177155914568914,
      "gap": 3.832950911696411,
      "truth": 4.0
    }
  ],
  "standardized": {
    "ATE": 2.7133370845321005,
    "ATT": 3.0292554386706074,
    "ATU": 2.471752460779125
  },
  "truth": {
    "ATE": 2.6977777777777776,
    "ATT": 3.0846153846153848,
    "ATU": 2.4019607843137254
  },
  "raw_gap": 6.505535014302103,
  "weights": {
    "ATE": [
      0.4922222222